In [1]:
"""

kaggle: liza_fomenko

Сначала я попробовала обойти splink и написать более-менее manual-created index

я попробовала SymSpell, но он работал очень долго (~1 час) + для адекватной настройки его параметров требовалось бы много запусков (потому что на скармливании части данных
результат будет нерепрезентативным). Поэтому от его использования после первого прогона я отказалась

Потом я решила использовать MinHashLSH. В принципе проблема со временем никуда не ушла - работало тоже в течение часа для всего набора данных. Здесь еще была проблема в том, что
я хотела на всех выдаваемых кандидатов считать расстояние левенштейна (для полной строки = row) и выбирать лучшего, но при маленьких порогах thr их было слишком много,
а при маленьких - слишком мого кандидатов, которые выдавались не сортировано, поэтому поиск работал приличное количество времени :( Ну и выбить получилось только 0.09, что довольно печально

ну и с HNGW не лучше получилось, потому что я все еще юзала MinHash, что было долго.... а с vectorizers моя борьба закончилась в самом начале.

Классы для них я оставила, ну просто чтоб не терялись. Они в конце

Собственно поэтому я перешла к splink. Для его написания я юзала собственно документацию с примерами + вот этот колаб (но в нем пример из документации)
https://colab.research.google.com/github/moj-analytical-services/splink/blob/master/docs/demos/examples/duckdb/link_only.ipynb#scrollTo=-KefN_4FWMWP
https://github.com/moj-analytical-services/splink

Первый запуск splink мне дал 0.26 на лмдерборде, я брала [block_on("substr(name, 1, 5)", "passport"), block_on("surname", "gender")] практически наугад, так как это был первый запуск + как
мне показалось потом у меня не зафитилось что-то (неправильно обработала)
Потом началась игра с block_on, cl.Comparison методами, threshold-ами и тд.

Обзятельный бейзлайн побился на  [["inn", "gender", "substr(surname, 1, 6)"], ["passport", "gender", "substr(name, 1, 5)"], ["birthdate", "gender"]], threshold=0.9; скор 0.8157


"""

'\nСначала я попробовала обойти splink и написать более-менее manual-created index\n\nя попробовала SymSpell, но он работал очень долго (~1 час) + для адекватной настройки его параметров требовалось бы много запусков (потому что на скармливании части данных\nрезультат будет нерепрезентативным). Поэтому от его использования после первого прогона я отказалась\n\nПотом я решила использовать MinHashLSH. В принципе проблема со временем никуда не ушла - работало тоже в течение часа для всего набора данных. Здесь еще была проблема в том, что\nя хотела на всех выдаваемых кандидатов считать расстояние левенштейна (для полной строки = row) и выбирать лучшего, но при маленьких порогах thr их было слишком много,\nа при маленьких - слишком мого кандидатов, которые выдавались не сортировано, поэтому поиск работал приличное количество времени :( Ну и выбить получилось только 0.09, что довольно печально\n\nну и с HNGW не лучше получилось, потому что я все еще юзала MinHash, что было долго.... а с vect

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

%cd /content/drive/My Drive/AI MASTERS/ML1/HW/HW6

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/My Drive/AI MASTERS/ML1/HW/HW6


In [3]:
import pandas as pd


# download data
orcs = pd.read_parquet('orcs.parquet')
empl = pd.read_parquet('employees.parquet')

In [4]:
# assert we have the same df logic

In [5]:
orcs.head()

,name,surname,fathername,gender,birthdate,inn,passport
0,габ`узхун,геля,шщэиевич,м,1961-10-13,209526320016,9427048895
1,сигурд,клабунов,васиолвич,м,1971-04-08,462095945786,9940565631
2,влоидмир,субботин,иванович,м,1974-07-22,611477616027,3851497038
3,реден,фарашкин,амрос`евич,м,<NA>,840087716808,5805184207
4,данислав,белухо,хльес,м,1962-11-11,<NA>,3287232005


In [6]:
empl.head()

,name,surname,fathername,gender,birthdate,inn,passport
0,жумавой,волоколамц*в,садагет оглы,м,1973-12-24,804926960301,7880355409
1,мубн,невмятуллин,"али,вич",м,1984-07-06,111068355926,1411814346
2,"ан,рей",карпоэ,вяч.славовеч,м,1962-03-28,617713534516,3428889207
3,олрга,рубцова,викторовна,ж,1961-10-13,530330958056,5497629378
4,сергей,леусенко,григорьевич,м,1998-09-10,872242822954,8988513519


In [7]:
# во-первых, ищем реальных орков среди работников. Если эти люди у меня не будут находиться, то что то точно работает не так

In [8]:
orcs_in_empl = pd.merge(empl, orcs, on=list(empl.columns), how='inner')
orcs_in_empl.shape

(37, 7)

In [9]:
# use splink

In [ ]:
!pip install splink

In [11]:
!pip install duckdb

In [12]:
# во-первых, splink заявляют, что строки должны идти без знаков препинания, поэтому от них придется избавляться во время предобработки

# 5 minutes

In [13]:
import re
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.comparison_library import CustomComparison
import duckdb
import pandas as pd
import numpy as np

class MySlipnkIndex:
    def __init__(self, empl, orcs, blocks_on, threshold=0.85):
        self.empl = empl.copy()
        self.orcs = orcs.copy()
        self.threshold = threshold
        self.blocks_on = []
        for pair in blocks_on:
            if len(pair) == 2:
                self.blocks_on.append(block_on(pair[0], pair[1]))
            elif len(pair) == 1:
                self.blocks_on.append(block_on(pair[0]))
            else:
                self.blocks_on.append(block_on(pair[0], pair[1], pair[2]))


        # Preprocess
        print('Preprocess data....')
        self._preprocess(self.empl)
        self._preprocess(self.orcs)
        print('Preprocessing done successfully!')

        print('Init settings and linker...')
        print(self.empl.head())


        # now we got a normal that can be processed by splink

        # create splink settings

        print('Building linker...')

        settings = SettingsCreator(
        link_type="link_only",
        blocking_rules_to_generate_predictions=self.blocks_on,
        comparisons=[
            cl.NameComparison("name"),
            cl.NameComparison("surname"),
            cl.NameComparison("fathername"),
            cl.NameComparison("passport"),
            cl.NameComparison("inn"),
            cl.ExactMatch("gender"),
            cl.DateOfBirthComparison("birthdate", input_is_string = False)])

        con = duckdb.connect(":memory:")
        con.execute("SET threads TO 16")


        self.linker = Linker(
        [self.orcs, self.empl],
        settings,
        db_api=DuckDBAPI(con),
        input_table_aliases=["orcs", "empl"]
        )

        print('Building linker completed successfully!')


    def _process_text(self, string):
        # if we got a nan value -> change to ''
        if pd.isna(string):
            return ""
        # remove everything that is not a russian letter
        return "".join(re.findall(r"[А-Яа-яёЁ]", string)).lower()

    def _process_digits(self, number):
        # if we got a nan value -> change to ''
        if pd.isna(number):
            return ""
        # remove everything that is not a digits
        return "".join(re.findall(r"\d", str(number)))

    def _preprocess(self, df):
        # splink qoute: "input data should be standardised, with consistent column names and formatting (e.g., lowercased, punctuation cleaned up, etc.)."
        # so here i will remove punctuation signs and convert all leters into lowercase

        cols_with_text_data = ['name', 'surname', 'fathername', 'gender']
        for col in cols_with_text_data:
            df[col] = df[col].apply(self._process_text)
        cols_with_numbers = ['passport', 'inn']
        for col in cols_with_numbers:
            df[col] = df[col].apply(self._process_digits)

        df['birthdate'] = pd.to_datetime(df['birthdate'], format='%Y-%m-%d', errors='coerce')

        df['full_name'] = df['surname'] + " " + df['name'] + " " + df['fathername']

        # add unique id which is just df index
        df['unique_id'] = df.index


    def train(self):
        self.linker.training.estimate_probability_two_random_records_match(self.blocks_on, recall=0.8)
        self.linker.training.estimate_u_using_random_sampling(max_pairs=int(2e6), seed=57)
        for block in self.blocks_on:
            self.linker.training.estimate_parameters_using_expectation_maximisation(block)

    def build_result(self):
        results = self.linker.inference.predict(threshold_match_probability=self.threshold)
        results = results.as_pandas_dataframe()
        print(f'got {results.shape[0]} orcs predictions')
        return results


    def generate_submission(self, df, subm_name = 'submission.parquet'):
        orc_candidates = df['unique_id_l']

        res = pd.DataFrame({
            'orig_index': orc_candidates.astype(np.uint64),
        }).reset_index(names='id')
        res.to_parquet(subm_name, index=False)





In [ ]:
splink_index = MySlipnkIndex(empl, orcs, [["inn", "gender", "substr(surname, 1, 6)"], ["passport", "gender", "substr(name, 1, 5)"], ["birthdate", "gender"]], threshold=0.9)
splink_index.train()
result = splink_index.build_result()

In [15]:
result.head()

,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,name_l,name_r,gamma_name,surname_l,...,inn_l,inn_r,gamma_inn,gender_l,gender_r,gamma_gender,birthdate_l,birthdate_r,gamma_birthdate,match_key
0,28.608277,1.000000,empl,orcs,483212,47518,мирзафан,мирзафан,4,инсурефелу,...,849960368524,,0,м,м,1,1978-05-14,1978-05-14,5,1
1,19.282683,0.999998,empl,orcs,483478,44600,мартин,мартин,4,русков,...,206103069228,206103069282,3,м,м,1,1986-11-16,1986-11-16,5,1
2,22.936129,1.000000,empl,orcs,484195,47085,евгения,евгени,3,кучеренко,...,693618035685,693618035685,4,ж,ж,1,1973-09-04,NaT,-1,1
3,12.572394,0.999836,empl,orcs,485048,13485,александр,александр,4,брыин,...,430998593526,,0,м,м,1,1967-02-25,1967-02-25,5,1
4,22.005180,1.000000,empl,orcs,485795,12447,вячеслав,вячеслов,3,рязонов,...,,304808338304,0,м,м,1,1971-02-22,1971-02-22,5,1


In [16]:
splink_index.generate_submission(result, 'submission_splink_6.parquet')

## first try - SymSpell for full_names

TOO LONG + optimization of params will take some time i guess

In [ ]:
def create_full_name(df):
    return df["name"].fillna('').astype(str) + " " + df["surname"].fillna('').astype(str) + " " + df["fathername"].fillna('').astype(str)

empl['full_name'] = create_full_name(empl)
orcs['full_name'] = create_full_name(orcs)

In [ ]:
empl.shape, orcs.shape

((2011759, 10), (47633, 10))

In [ ]:
empl.head()

,name,surname,fathername,gender,birthdate,inn,passport,full_name
0,жумавой,волоколамц*в,садагет оглы,м,1973-12-24,804926960301,7880355409,жумавой волоколамц*в садагет оглы
1,мубн,невмятуллин,"али,вич",м,1984-07-06,111068355926,1411814346,"мубн невмятуллин али,вич"
2,"ан,рей",карпоэ,вяч.славовеч,м,1962-03-28,617713534516,3428889207,"ан,рей карпоэ вяч.славовеч"
3,олрга,рубцова,викторовна,ж,1961-10-13,530330958056,5497629378,олрга рубцова викторовна
4,сергей,леусенко,григорьевич,м,1998-09-10,872242822954,8988513519,сергей леусенко григорьевич


In [ ]:
# first try --- no customization, just SymSpell (rather long, >1min per 1k of queries => total ~1 hour (crazy))

In [ ]:
from symspellpy import SymSpell, Verbosity


# first try: try to print all nearest people baset on their names
class MyFunnyIndexJustSymSpell:
    def __init__(self, max_edit_distance=2, prefix_length=7):
        self.sym = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=prefix_length)
        self.name_to_idx = {}

    def build_index(self, fullnames):
        for idx, full_name in enumerate(fullnames):
            self.sym.create_dictionary_entry(full_name, 1)
            self.name_to_idx[full_name] = idx

    def search(self, orcs_names, k=1):
        suggestions = []
        idxs = []

        for num, query in enumerate(orcs_names):
            if num %1000 == 0:
                print(num)
            suggestion = self.sym.lookup(query, Verbosity.TOP, max_edit_distance=2)
            if suggestion:
                suggestion = suggestion[0].term
                suggestions.append(suggestion)
                idxs.append(self.name_to_idx[suggestion])
            else:
                suggestions.append('NONE')
                idxs.append(-1)


        return suggestions, idxs



"""
def my_metric(...):
   ...
"""

index = MyFunnyIndexJustSymSpell()
index.build_index(empl['full_name'])
orc_candidates, idxes = index.search(orcs['full_name'], k=1)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000


In [ ]:
new_df = orcs

In [ ]:
new_df['candidate_name'] = orc_candidates
new_df['candidate_idx'] = idxes

In [ ]:
new_df.head(20)

,name,surname,fathername,gender,inn,passport,year,month,date,full_name,candidate_name,candidate_idx
0,габ`узхун,геля,шщэиевич,м,209526320016,9427048895,1961,10,13,габ`узхун геля шщэиевич,NONE,-1
1,сигурд,клабунов,васиолвич,м,462095945786,9940565631,1971,04,08,сигурд клабунов васиолвич,NONE,-1
2,влоидмир,субботин,иванович,м,611477616027,3851497038,1974,07,22,влоидмир субботин иванович,влодимир субботин иванович,1877530
3,реден,фарашкин,амрос`евич,м,840087716808,5805184207,<NA>,<NA>,<NA>,реден фарашкин амрос`евич,NONE,-1
4,данислав,белухо,хльес,м,<NA>,3287232005,1962,11,11,данислав белухо хльес,NONE,-1
5,татьяна,клюжини,юръевно,ж,<NA>,1787563251,1975,12,25,татьяна клюжини юръевно,NONE,-1
6,ночмидни,брегвадзе,фазилоглы,м,652621888893,<NA>,<NA>,<NA>,<NA>,ночмидни брегвадзе фазилоглы,NONE,-1
7,мадаменжон,исыамгареев,осенахулович,м,<NA>,4545494459,1971,01,26,мадаменжон исыамгареев осенахулович,NONE,-1
8,селимжан,беккерев,адезжонвоич,м,<NA>,<NA>,<NA>,<NA>,<NA>,селимжан беккерев адезжонвоич,NONE,-1
9,алаверды,баламутов,геннаиевия,м,565678806882,4808625499,1967,04,04,алаверды баламутов геннаиевия,NONE,-1


In [ ]:
new_df.shape

(47633, 12)

In [ ]:
new_df[new_df['candidate_idx'] != -1].shape

(4144, 12)

In [ ]:
res = pd.DataFrame({
    'orig_index': idxes.astype(np.uint64),
}).reset_index(names='id')
res.to_parquet('submission.parquet', index=False)


In [ ]:
#   слишком сильные условия поставила?..... возможно...

#   энивэй слишком долго работает

In [ ]:
# we need faster solution => first of all solve for texts

# tokenization -> convert full_name into vec / hash / any other numerical representation
# examples: MinHash, SimHash, sliding hash?

# example -> get k nighbours -> build SimSpell / select best on metric / best on Levenstein distance -> orc candidates
# also we can see the length of real orcs))))) as if we have 2 dfs intersection, submitting this solution gives J = 37 / X

## try LSH from data in a row

TOO long + only 0.09 on leaderboard


In [ ]:
# example: TRY just full_name, i wanna make LHS ->query -> 50 candidates -> SimSpell -> query -> best candidate

In [ ]:
# try to implement just LSH - i cant use all rows here. so i am to combine everything in a single row?..

In [ ]:
!pip install vectorizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 3.6 MB/s eta 0:00:00


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 81.0 MB/s eta 0:00:00


In [ ]:
!pip install rapidfuzz

In [ ]:
!pip install datasketch

In [ ]:
!pip install symspellpy

In [ ]:
from datasketch import MinHashLSH, MinHash
from rapidfuzz.distance import Levenshtein


class MyFunnyIndex:
    def __init__(self, df, num_perm=128, thr = 0.5, n = 3):
        self.num_perm = num_perm
        self.df = df
        self.thr = thr
        self.n = n


        print('start Building MinHash LSH...')

        self.index_lsh = MinHashLSH(threshold = self.thr, num_perm = self.num_perm)

        # make str from df
        self._make_row_str(self.df)


        # apply MinHash for every full_str generated by the func above and add it to the MinHashLSH
        for idx, full_str in enumerate(self.df['full_str']):
            minhash = self._apply_MinHash(full_str)
            self.index_lsh.insert(idx, minhash)               # insert object into LSH

        print('Building MinHash LSH finished successfully')

    def _make_row_str(self, df):
        df['full_str'] = df["name"].fillna('').astype(str) + " " + df["surname"].fillna('').astype(str) + " " + df["fathername"].fillna('').astype(str) + " " + df["gender"].fillna('').astype(str)

    def _make_ngrams_for_str(self, string):
        string = str(string)
        return {string[i:i + self.n] for i in range(len(string) - self.n + 1)}


    def _apply_MinHash(self, string):
        minhash = MinHash(num_perm=self.num_perm)
        ngrams = list(self._make_ngrams_for_str(string))
        minhash.update_batch([ng.encode("utf8") for ng in ngrams])
        return minhash


    def search(self, q_df, return_all = False):

        if isinstance(q_df, str):
            candidates_ids = self._search_one_str(q_df)

            if return_all:
                return candidates_ids
            else:
                best_candidates_ids = self._find_best(candidates_ids, q_df)
                return best_candidates_ids


        # make str from df
        self._make_row_str(q_df)

        candidates_ids = []

        # find best for every
        for _, row in q_df.iterrows():
            full_str = row['full_str']
            cand_ids = self._search_one_str(full_str)

            # if no empl found for the orc
            if not cand_ids:
                candidates_ids.append(-1)


            elif return_all:
                candidates_ids.append(cand_ids)
            else:
                best_candidates_ids = self._find_best(cand_ids, full_str)
                candidates_ids.append(best_candidates_ids)

        return candidates_ids


    def _search_one_str(self, full_str):
        minhash = self._apply_MinHash(full_str)                             # got minhash
        candidates_ids = [int(id) for id in self.index_lsh.query(minhash)]
        return candidates_ids

    def _find_best(self, candidates_ids, full_str):
        all_lev = []
        for cand in candidates_ids[:50]:
            cand_full_str = self.df.iloc[cand, :]['full_str']
            all_lev.append([cand, Levenshtein.distance(full_str, cand_full_str)])

        all_lev.sort(key=lambda x: x[1])

        best_ind = all_lev[0][0]

        return best_ind






In [ ]:
from datasketch import MinHashLSH, MinHash
from rapidfuzz.distance import Levenshtein


class MyFunnyIndex:
    def __init__(self, df, num_perm=128, thr = 0.5, n = 3):
        self.num_perm = num_perm
        self.df = df
        self.thr = thr
        self.n = n


        print('start Building MinHash LSH...')

        self.index_lsh = MinHashLSH(threshold = self.thr, num_perm = self.num_perm)

        # make str from df
        self._make_row_str(self.df)


        # apply MinHash for every full_str generated by the func above and add it to the MinHashLSH
        for idx, full_str in enumerate(self.df['full_str']):
            minhash = self._apply_MinHash(full_str)
            self.index_lsh.insert(idx, minhash)               # insert object into LSH

        print('Building MinHash LSH finished successfully')

    def _make_row_str(self, df):
        df['full_str'] = df["name"].fillna('').astype(str) + " " + df["surname"].fillna('').astype(str) + " " + df["fathername"].fillna('').astype(str) + " " + df["gender"].fillna('').astype(str) + " " + df["birthdate"].fillna('').astype(str) + " " + df["passport"].fillna('').astype(str)

    def _make_ngrams_for_str(self, string):
        string = str(string)
        return {string[i:i + self.n] for i in range(len(string) - self.n + 1)}


    def _apply_MinHash(self, string):
        minhash = MinHash(num_perm=self.num_perm)
        ngrams = list(self._make_ngrams_for_str(string))
        minhash.update_batch([ng.encode("utf8") for ng in ngrams])
        return minhash


    def search(self, q_df, return_all = False):

        if isinstance(q_df, str):
            candidates_ids = self._search_one_str(q_df)

            if return_all:
                return candidates_ids
            else:
                best_candidates_ids = self._find_best(candidates_ids, q_df)
                return best_candidates_ids


        # make str from df
        self._make_row_str(q_df)

        candidates_ids = []

        # find best for every
        for _, row in q_df.iterrows():
            full_str = row['full_str']
            cand_ids = self._search_one_str(full_str)

            # if no empl found for the orc
            if not cand_ids:
                candidates_ids.append(-1)


            elif return_all:
                candidates_ids.append(cand_ids)
            else:
                best_candidates_ids = self._find_best(cand_ids, full_str)
                candidates_ids.append(best_candidates_ids)

        return candidates_ids


    def _search_one_str(self, full_str):
        minhash = self._apply_MinHash(full_str)                             # got minhash
        candidates_ids = [int(id) for id in self.index_lsh.query(minhash)]
        return candidates_ids

    def _find_best(self, candidates_ids, full_str):
        all_lev = []
        for cand in candidates_ids[:50]:
            cand_full_str = self.df.iloc[cand, :]['full_str']
            all_lev.append([cand, Levenshtein.distance(full_str, cand_full_str)])

        all_lev.sort(key=lambda x: x[1])

        best_ind = all_lev[0][0]

        return best_ind






In [ ]:
small_empl = empl.iloc[:100000, :]

In [ ]:
index = MyFunnyIndex(small_empl, num_perm=128, thr = 0.6, n = 3)

start Building MinHash LSH...


/tmp/ipython-input-483295873.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['full_str'] = df["name"].fillna('').astype(str) + " " + df["surname"].fillna('').astype(str) + " " + df["fathername"].fillna('').astype(str) + " " + df["gender"].fillna('').astype(str)


Building MinHash LSH finished successfully


In [ ]:
orcs_ids = index.search(orcs, return_all = False)

In [ ]:
orcs_ids[:50]

[-1,
 38240,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 72796,
 -1,
 -1,
 -1,
 60076,
 -1,
 -1,
 -1,
 20077,
 -1,
 -1,
 -1,
 92637,
 -1,
 -1,
 64097,
 -1,
 -1,
 -1,
 -1,
 -1,
 12318,
 -1,
 -1,
 -1,
 -1,
 -1,
 -1,
 84680,
 -1,
 37623,
 -1,
 14471,
 -1,
 -1,
 21043]

In [ ]:
import numpy as np

orcs_ids = np.array(orcs_ids)
orc_candidates = orcs_ids[orcs_ids != -1]

In [ ]:
res = pd.DataFrame({
    'orig_index': orc_candidates.astype(np.uint64),
}).reset_index(names='id')
res.to_parquet('submission.parquet', index=False)


In [ ]:
orcs.iloc[3, :]

,3
name,реден
surname,фарашкин
fathername,амрос`евич
gender,м
birthdate,<NA>
inn,840087716808
passport,5805184207
full_str,реден фарашкин амрос`евич м 5805184207


In [ ]:
empl.iloc[1314829, :]

,1314829
name,гутиерес
surname,шандарин
fathername,файршевеч
gender,м
birthdate,1969-12-01
inn,413316979366
passport,None
full_str,гутиерес шандарин файршевеч м 1969-12-01
